In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df.isnull().sum()

In [ ]:
columnsDel=[]
null_threshold = 0.90 * len(df)

for col in df.columns:

    if df[col].isnull().sum() >= null_threshold:
        columnsDel.append(col)

print('columns with >90% missing:',columnsDel)


In [ ]:
df= df.drop(columns=columnsDel)

In [ ]:
# rest of null
for x in df.columns :
 df[x] =df[x].fillna(df[x].mean())

In [ ]:
df.isnull().sum().sum()

In [ ]:
# Task 2: Write your code here:
#check and remove duplicates if any exist
df.duplicated().sum()

In [ ]:
# Task 3: Write your code here:
df.info() # no cat columns

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here:
#Check for target imbalance and state if it is imbalanced or not
df['Target'].value_counts() #imbalance

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Task 1: Write your code here:
y = df['Target']
X = df.drop(columns=['Target'])

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
from catboost import CatBoostClassifier


In [ ]:
# Task 2,3,4,5: Write your code here:
#StratifiedKFold
n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
model =CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

In [ ]:
accuracy_l =[]
f1_l =[]

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy=accuracy_score(y_test, y_pred)
  print(accuracy)

  f1 =f1_score(y_test, y_pred, zero_division=0)
  print(f1)
  accuracy_l.append(accuracy)
  f1_l.append(f1)

In [ ]:
accuracy_l

In [ ]:
# avrage acc
t=0.0
for x in accuracy_l :
  t+=x

print('accuracy avg' , t/6)

In [ ]:
f1_l

In [ ]:
# avrage acc
t=0.0
for x in f1_l :
  t+=x

print('f1avg' , t/6)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Task 1: Write your code here:
feature_importance = (
    pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': model.feature_importances_
    })
    .sort_values(by='Importance', ascending=False)
    .reset_index(drop=True)
)

display(feature_importance.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance.head(15),
    palette='viridis'
)
plt.title(' — Top 15 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
 # p_2 is most impotent feature
print(feature_importance.head(1))